### VaspInput代码测试

In [ ]:
import os
from pathlib import Path
from pymatgen.io.vasp.inputs import Incar
from pymatgen.core import Structure


class Vasp_check:
    """
    VASP输入文件的校验器。
    用于自动化检查INCAR参数、MAGMOM设定以及DFT+U参数是否符合预期。
    """
    
    def __init__(self, calc_dir):
        self.calc_dir = Path(calc_dir)
        self.incar_path = self.calc_dir / "INCAR"
        self.poscar_path = self.calc_dir / "POSCAR"
        
        #1.检查文件是否存在
        assert self.incar_path.exists(), f"找不到 INCAR: {self.incar_path}"
        assert self.poscar_path.exists(), f"找不到 POSCAR: {self.poscar_path}"
        
        #2.读取文件
        self.incar = Incar.from_file(self.incar_path)
        self.structure = Structure.from_file(self.poscar_path)
        
        #3.提取属性, 结构的原子数及其严格的顺序
        self.total_atoms = len(self.structure)
        self.element_types = [sp.symbol for sp in self.structure.types_of_specie]
        self.num_elements = len(self.element_types)
        

    def get_parameter(self, param_name, default=None):
        """获取参数接口"""
        return self.incar.get(param_name, default)

    def check_exact_params(self, expected_params: dict):
        """
        通用检查：批量检查需要精确匹配的参数
        ：excepted_params: 预期的参数字典，例如{"IBRION": 2, "ISIF": 3}
        """
        for key, expected_val in expected_params.items():
            actual_val = self.incar.get(key)
            assert actual_val is not None, f"INCAR 中缺失参数: {key}"
            
            if isinstance(expected_val, bool):
                # pymatgen 通常会解析为 bool，但如果解析成了字符串，我们需要手动识别
                if isinstance(actual_val, str):
                    # 涵盖 VASP 常见的布尔值写法
                    is_true = actual_val.strip().upper() in [".TRUE.", "TRUE", "T"]
                    actual_bool = is_true
                else:
                    actual_bool = bool(actual_val)
                    
                assert actual_bool == expected_val, \
                    f"{key} 布尔值错误！预期为 {expected_val}, 实际为 {actual_val}"

            elif isinstance(expected_val, (int, float)):
                try:
                    actual_num = float(actual_val)
                    expected_num = float(expected_val)
                except (ValueError, TypeError):
                    raise AssertionError(f"{key} 类型错误！无法将实际值 '{actual_val}' 转换为数值。")

                #容差比较
                assert abs(actual_num - expected_num) < 1e-5, \
                    f"{key} 数值错误！预期为 {expected_val}, 实际为 {actual_val}"

            elif isinstance(expected_val, str):
                #统一转大写并去除首尾空格后比较
                actual_str = str(actual_val).strip().upper()
                expected_str = expected_val.strip().upper()
                
                assert actual_str == expected_str, \
                    f"{key} 字符串错误！预期为 '{expected_val}', 实际为 '{actual_val}'"
            else:
                assert actual_val == expected_val, \
                    f"{key} 设置错误！预期为 {expected_val}, 实际为 {actual_val}"
            
        print(f"常规参数精确匹配校验通过！({list(expected_params.keys())})")


    def check_magmom(self, expected_magmom_dict=None):
        """
        检查 MAGMOM 设置
        :param expected_magmom_dict: 预期的元素磁矩字典，如 {"V": 3.0, "O": 0.6}
        """
        magmoms = self.incar.get("MAGMOM")
        
        assert magmoms is not None, "INCAR 中缺失 MAGMOM 参数！"
        if not isinstance(magmoms, list):
            magmoms = [magmoms]
        assert len(magmoms) == self.total_atoms, \
            f"MAGMOM 长度 ({len(magmoms)}) 与 POSCAR 总原子数 ({self.total_atoms}) 不匹配！"
    
        if expected_magmom_dict is not None:
            for i, site in enumerate(self.structure):
                element = site.specie.symbol
                expected_val = expected_magmom_dict.get(element, 0.0) 
                actual_val = magmoms[i]
            
                assert abs(actual_val - expected_val) < 1e-4, \
                f"MAGMOM 数值错误！第 {i+1} 个原子 ({element}) 为 {actual_val}，预期为 {expected_val}。"
            print(f"MAGMOM 长度和数据校验通过！共 {self.total_atoms} 个原子")
        print(f"MAGMOM 长度校验通过！未检查具体数值，共 {self.total_atoms} 个原子")

    def check_ldau(self, expected_u_settings=None):
        """
        检查 DFT+U (LDA+U) 设置
        :param expected_u_settings: 预期的 U 值字典
        """
        assert self.incar.get("LDAU") is True, "INCAR 中未开启 LDAU = True"
        
        # 1. 基础校验：检查 U 相关的数组参数长度是否等于元素种类数
        for param in ["LDAUL", "LDAUU", "LDAUJ"]:
            actual_list = self.incar.get(param)
            assert actual_list is not None, f"INCAR 中缺失 {param} 参数！"
            
            if not isinstance(actual_list, list):
                actual_list = [actual_list]
                
            assert len(actual_list) == self.num_elements, \
                f"{param} 长度 ({len(actual_list)}) 与元素种类数 ({self.num_elements}) 不匹配！"

        # 2. 深度校验：具体数值（仅在传入预期字典时执行）
        if expected_u_settings is not None:
            # 检查标量参数
            for param in ["LDAUTYPE", "LDAUPRINT"]:
                if param in expected_u_settings:
                    actual_val = self.incar.get(param)
                    expected_val = expected_u_settings[param]
                    assert actual_val == expected_val, f"{param} 错误！预期 {expected_val}，实际 {actual_val}。"

            # 检查数组参数具体数值 (严格按照 POSCAR 元素顺序)
            for param in ["LDAUL", "LDAUU", "LDAUJ"]:
                if param not in expected_u_settings:
                    continue
                    
                actual_list = self.incar.get(param)
                if not isinstance(actual_list, list):
                    actual_list = [actual_list]
                expected_dict = expected_u_settings[param]
                    
                for i, element in enumerate(self.element_types):
                    expected_val = expected_dict.get(element, 0)
                    actual_val = actual_list[i]
                    
                    assert abs(float(actual_val) - float(expected_val)) < 1e-4, \
                        f"{param} 错误！元素 {element} 的值为 {actual_val}，预期为 {expected_val}。"
            print(f"DFT+U 长度与数值校验通过！(元素顺序: {' '.join(self.element_types)})")
        else:
            print(f"DFT+U 长度校验通过！(未检查具体数值，元素顺序: {' '.join(self.element_types)})")

##KPOINTS和script的设定还没有写





### Bulk写入方法
参数默认,其余参考MPMetalRelax  
"""    
"EDIFFG": -0.02,  
"EDIFF": 1E-6,  
"POTIM": 0.20,  
"ENCUT": 520,  
"IBRION": 2,  
"LORBIT": 10,  
"NSW": 500,  
"LREAL": "Auto",  
"""  

In [17]:
import vasp_inputs
from vasp_inputs import VaspInputMaker
from pathlib import Path

#使用文件夹形式，自动读取文件夹中的结构文件，优先使用CONTCAR/POSCAR/.vasp/.cif
structure_bulk = "/data2/home/luodh/Test/VaspInput-Test/Bulk-calc" 
Test_result_dir = Path(structure_bulk) / "Bulk-optim"
magmom = {"V": 3.0, "O": 0.6, "Cr": 3.0}
user_incar_settings = {"IBRION": 2, "MAGMOM":magmom}#特殊化INCAR参数的设置
Test1_bulk = VaspInputMaker(user_incar_settings=user_incar_settings)
results = Test1_bulk.write_bulk(output_dir= Test_result_dir, 
                                structure=structure_bulk)

Test_incar = Path(Test_result_dir).resolve() / "INCAR"

check  = Vasp_check(Test_result_dir)
check.check_exact_params(expected_params={"IBRION":2, "ISIF":3, "ISPIN": 2, "LDAU": True})
check.check_ldau() #常规检查参数是否设置正确，顺序和个数
#检查具体的设置，数值加个数，LDAUU,LDAUUL,LDAUUJ均可按照这个方法检查
check.check_ldau(expected_u_settings={"LDAUU":{"V": 3.25, "O": 0, "Cr": 3.7}})
#检查具体的设置
check.check_magmom(expected_magmom_dict={"V": 3.0 , "O": 0.6, "Cr": 3.0}) #

常规参数精确匹配校验通过！(['IBRION', 'ISIF', 'ISPIN', 'LDAU'])
DFT+U 长度校验通过！(未检查具体数值，元素顺序: V Cr O)
DFT+U 长度与数值校验通过！(元素顺序: V Cr O)
MAGMOM 长度和数据校验通过！共 56 个原子
MAGMOM 长度校验通过！未检查具体数值，共 56 个原子
